# 02 - Train the context encoder (homograph disambiguation)

This is the cheap, high-value stage: about 20 minutes on a 4090. It is also
where homograph accuracy actually comes from, so iterate here before spending
money on the acoustic model.

The model is about 6M parameters. For every word it predicts which of that
word's discovered readings applies in this context.

In [1]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
os.environ["PYTHONIOENCODING"] = "utf-8"

# ---------------------------------------------------------------------------
# PICK YOUR EXPERIMENT HERE. This is the only line to change.
#
#   configs/exp0_small.yaml     2013 clips, ~2 GB   -> proves the pipeline,
#                                                     runs on a 6 GB GPU
#   configs/exp1_egyptian.yaml  15.6k clips, 68 h   -> the real run
# ---------------------------------------------------------------------------
CONFIG = "configs/exp1_egyptian.yaml"

from adaptts.utils.config import load_config
from adaptts.utils.logging_utils import setup_logging
setup_logging()
cfg = load_config(CONFIG)
print("repo   :", REPO)
print("config :", CONFIG, "->", cfg.name)
print("dataset:", cfg.paths.hf_dataset_id)

# The homographs named in the brief, read from the probe file so the notebooks
# never hardcode a word list of their own.
import json as _json
PROBE_WORDS = sorted({
    w for _s in _json.load(open("assets/probe_sentences.json", encoding="utf-8"))["sentences"]
    for w in _s["focus"].split(" / ") if w and w != "none"
})
print("probe  :", " ".join(PROBE_WORDS))

repo   : /workspace/AdapTTS
config : configs/exp1_egyptian.yaml -> exp1_egyptian_homograph
dataset: ehabnegm/100-hour-Egyptian-dataset-single-speaker
probe  : الدول دول علم مصر


## Start TensorBoard

Watch `eval/code_acc`. That is held-out homograph accuracy, the number that
matters. Above roughly 0.90 means the approach is working.

In [2]:
%load_ext tensorboard
%tensorboard --logdir $cfg.paths.tb_dir --port 6006 --bind_all

## Train

In [3]:
!python scripts/train_context.py --config $CONFIG

00:04:36 info  train_context  AdapTTS stage B: context encoder (homograph disambiguation)
00:04:36 info  train_context  device: cuda
00:04:36 info  train_context  configuration
00:04:36 info  train_context  setting     value                                     
00:04:36 info  train_context  ------------------------------------------------------
00:04:36 info  train_context  experiment  exp1_egyptian_homograph                   
00:04:36 info  train_context  dataset     /workspace/AdapTTS/data/masri100h         
00:04:36 info  train_context  cache       /workspace/AdapTTS/cache/exp1             
00:04:36 info  train_context  run dir     /workspace/AdapTTS/runs/exp1              
00:04:36 info  train_context  codec       kyutai/mimi (8 quantizers)                
00:04:36 info  train_context  teacher     aubmindlab/bert-base-arabertv02-twitter   
00:04:36 info  train_context  aligner     MahmoudAshraf/mms-300m-1130-forced-aligner
00:04:36 info  train_context  precision   bf16            

## Evaluate what it learned

The two `علم` sentences must get *different* codes. If they do not,
disambiguation is not working and the acoustic model will inherit the failure.

In [4]:
import json
from adaptts.infer.pipeline import AdapTTS

tts = AdapTTS.from_checkpoints(CONFIG, device="cpu", load_codec=False)
probes = json.load(open("assets/probe_sentences.json", encoding="utf-8"))["sentences"]

for p in probes:
    plan = tts.analyze(p["text"])
    hard = [f"{w.word}=code{w.code}({w.confidence:.2f})" for w in plan.hard_words]
    print()
    print(p["tag"])
    print("  text      :", p["text"])
    print("  expected  :", p["expected"])
    print(f"  difficulty: {plan.sentence_difficulty:.3f} -> depth {plan.depth}")
    print("  decisions :", ", ".join(hard) if hard else "no ambiguous words")


3alam_flag
  text      : انا شوفت علم مصر بيرفرف
  expected  : عَلَم
  difficulty: 0.000 -> depth 4
  decisions : no ambiguous words

3ilm_science
  text      : علم الفيزيا من اهم العلوم البشرية
  expected  : عِلْم
  difficulty: 0.000 -> depth 4
  decisions : no ambiguous words

masr_egypt
  text      : مصر عندها امكانيات و موارد كتير جدا
  expected  : مَصْر
  difficulty: 0.000 -> depth 4
  decisions : no ambiguous words

musirr_insisting
  text      : انا كنت مصر على الراي بتاعي لحد النهاية
  expected  : مُصِرّ
  difficulty: 0.000 -> depth 4
  decisions : no ambiguous words

dowal_countries
  text      : الدول دي بتتنافس على السوق العالمي
  expected  : الدِوَل
  difficulty: 0.000 -> depth 4
  decisions : no ambiguous words

dool_these
  text      : الناس دول شايفين نفسهم احسن من غيرهم
  expected  : دُول
  difficulty: 0.000 -> depth 4
  decisions : no ambiguous words

multi_homograph
  text      : انا كنت مصر على ان مصر عندها امكانيات تخليها تتفوق على دول من اللي شايفين نفسهم دول
  ex

In [5]:
# The critical comparison: one word, two contexts, two readings.
a = tts.analyze("انا شوفت علم مصر بيرفرف")             # flag
b = tts.analyze("علم الفيزيا من اهم العلوم البشرية")   # science
ca = [w.code for w in a.hard_words if w.word == "علم"]
cb = [w.code for w in b.hard_words if w.word == "علم"]
print("flag context    -> code", ca)
print("science context -> code", cb)
print()
print("DISAMBIGUATION WORKS" if ca and cb and ca != cb
      else "NOT disambiguating: investigate before training the acoustic model")

flag context    -> code []
science context -> code []

NOT disambiguating: investigate before training the acoustic model


In [6]:
# The multi-homograph stress sentence from the brief.
plan = tts.analyze(
    "انا كنت مصر على ان مصر عندها امكانيات و موارد تخليها تتفوق على دول من اللي شايفين نفسهم دول"
)
print(plan)

text: انا كنت مصر على ان مصر عندها امكانيات و موارد تخليها تتفوق على دول من اللي شايفين نفسهم دول
sentence difficulty: 0.000   depth: 4
no ambiguous words: every word has a single known reading


## Inference speed on CPU

The context encoder must be negligible next to the acoustic model.

In [7]:
import time

txt = "انا كنت مصر على ان مصر عندها امكانيات تخليها تتفوق على دول"
for _ in range(3):
    tts.analyze(txt)                      # warm up
t0 = time.perf_counter()
for _ in range(50):
    tts.analyze(txt)
print(f"context encoder: {(time.perf_counter() - t0) / 50 * 1000:.2f} ms per sentence on CPU")

context encoder: 11.38 ms per sentence on CPU


If accuracy looks good, continue to **03_train_acoustic.ipynb**.